# Goal of this substep:
- Confirm that NVML is available and readable on the GPU
- Do not log yet, do not train anything yet

In [10]:
!nvidia-smi


Wed Jan  7 15:19:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             29W /   70W |     263MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install NVML Python Binding

In [11]:
!pip install pynvml


Minimal NVML Power Read Test

In [12]:
from pynvml import *

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

power_mw = nvmlDeviceGetPowerUsage(handle)
power_watts = power_mw / 1000.0

print(f"Instantaneous GPU Power: {power_watts:.2f} W")

nvmlShutdown()


Instantaneous GPU Power: 29.54 W


### Design Rules 

- Sampling interval: 100 ms (0.1 s)

- Sampling type: time-based

- Logged fields:

   - Wall-clock timestamp (seconds)

   - Power (Watts)

Output: in-memory list

### Implement Minimal Sampling Loop

In [13]:
import time
from pynvml import *

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

samples = []
interval = 0.1  # 100 ms
duration = 5.0  # seconds

start_time = time.time()

while time.time() - start_time < duration:
    ts = time.time()
    power_watts = nvmlDeviceGetPowerUsage(handle) / 1000.0
    samples.append((ts, power_watts))
    time.sleep(interval)

nvmlShutdown()

print(f"Collected {len(samples)} samples")
print("First 5 samples:")
for s in samples[:5]:
    print(s)


Collected 49 samples
First 5 samples:
(1767799181.14978, 29.544)
(1767799181.252725, 29.528)
(1767799181.3550112, 29.528)
(1767799181.4572356, 29.528)
(1767799181.5595613, 29.626)


### Implement CSV Logger

In [14]:
import time
import csv
from pynvml import *

output_file = "gpu_power_log.csv"
interval = 0.1   # 100 ms
duration = 5.0   # seconds

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "power_watts"])

    start_time = time.time()
    while time.time() - start_time < duration:
        ts = time.time()
        power_watts = nvmlDeviceGetPowerUsage(handle) / 1000.0
        writer.writerow([ts, power_watts])
        time.sleep(interval)

nvmlShutdown()

print(f"Power log written to {output_file}")


Power log written to gpu_power_log.csv


In [15]:
!head gpu_power_log.csv


timestamp,power_watts
1767799186.1764982,29.429
1767799186.2814357,29.528
1767799186.383755,29.544
1767799186.485951,29.445
1767799186.5881665,29.528
1767799186.6903431,29.528
1767799186.792516,29.544
1767799186.894676,29.528
1767799186.9968724,29.544


### Graceful Start–Stop & Load Sensitivity Validation

Simple GPU Load Generator

We will generate artificial GPU load using a dummy tensor loop.

In [16]:
import torch
import time

device = torch.device("cuda")

x = torch.randn(4096, 4096, device=device)

start = time.time()
while time.time() - start < 5:
    x = torch.matmul(x, x)


In [ ]:
import time
import csv
import torch
from pynvml import *

output_file = "gpu_power_load_test.csv"
interval = 0.1
duration = 10.0  # total time

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "power_watts"])

    start_time = time.time()
    while time.time() - start_time < duration:
        ts = time.time()
        power_watts = nvmlDeviceGetPowerUsage(handle) / 1000.0
        writer.writerow([ts, power_watts])

        # Inject load between t=2s and t=7s
        if 2 < time.time() - start_time < 7:
            x = torch.randn(2048, 2048, device="cuda")
            x = torch.matmul(x, x)

        time.sleep(interval)

nvmlShutdown()

print("Load test logging complete.")


In [ ]:
import pandas as pd

df = pd.read_csv("gpu_power_load_test.csv")
df.plot(x="timestamp", y="power_watts")
